# Old vs New: Hopf grid search vs gradient descent (mean ± std over seeds)

Reads per-seed test-inter metrics from `results/runs/{ds}_{model_key}_seed{N}.json` for `model_key in {hopf_grid, hopf}` and plots the same nine-metric bar chart as section 4 of `empirical_vs_simulated_comparison.ipynb`, but with bars = mean across seeds and a vertical error bar = std across seeds.

Run the training jobs first (see `examples/old_vs_new_seeds.ipynb`'s docstring of `SEEDS` below for the seed list); each sbatch command writes one JSON.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import matplotlib.pyplot as plt

project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

plt.rcParams['figure.dpi'] = 140

In [ ]:
# Auto-create parent dirs for savefig (added for reproducible runs)
import matplotlib.figure as _mpl_figure
from pathlib import Path as _Path
_orig_savefig = _mpl_figure.Figure.savefig
def _patched_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, (str, _Path)):
        p = _Path(fname)
        if p.parent and not p.parent.exists():
            p.parent.mkdir(parents=True, exist_ok=True)
    return _orig_savefig(self, fname, *args, **kwargs)
_mpl_figure.Figure.savefig = _patched_savefig


In [ ]:
SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
DATASETS = ['ts_young']
DATASET_TITLES = {'ts_young': 'HCP Dataset'}
RESULTS_DIR = project_root / 'results' / 'runs'
OUT_BASE = project_root / 'paper_new'
for sub in ['images/comparison', 'images_png/comparison', 'images_svg/comparison']:
    (OUT_BASE / sub).mkdir(parents=True, exist_ok=True)

# (display_label, metric_key, higher_is_better)
# Keys must match the 'test_inter' keys in results/runs/*.json (see EVAL_METRIC_KEYS
# in src/training/evaluation.py).
METRICS = [
    ('FC ↑',     'fc_correlation',          True),
    ('FC MSE ↓', 'fc_mse',                  False),
    ('phFC ↑',   'phase_fc_correlation',    True),
    ('FCD ↓',    'fcd_ks',                  False),
    ('phFCD ↓',  'phfcd_ks',                False),
    ('Meta ↓',   'metastability_diff',      False),
    ('TS ↑',     'temporal_correlation',    True),
    ('PSD ↓',    'power_spectrum_distance', False),
    ('Auto ↓',   'autocorr_distance',       False),
]

OLD_KEY = 'hopf_grid'
NEW_KEY = 'hopf'
OLD_COLOR = '#9a9a9a'
NEW_COLOR = '#e48a6a'

In [ ]:
def load_seed_metrics(ds_tag: str, model_key: str, seeds=SEEDS, split: str = 'test_inter'):
    """Return ndarray (n_seeds, n_metrics) of values, NaN for missing files."""
    out = np.full((len(seeds), len(METRICS)), np.nan, dtype=float)
    for si, s in enumerate(seeds):
        path = RESULTS_DIR / f'{ds_tag}_{model_key}_seed{s}.json'
        if not path.exists():
            print(f'  missing: {path.relative_to(project_root)}')
            continue
        payload = json.loads(path.read_text())
        m = payload['metrics'][split]
        for mi, (_, key, _) in enumerate(METRICS):
            v = m.get(key)
            if v is not None:
                out[si, mi] = float(v)
    return out  # (S, M)


def mean_std(values: np.ndarray):
    """Mean and (sample) std over axis 0, ignoring NaNs."""
    mean = np.nanmean(values, axis=0)
    std = np.nanstd(values, axis=0, ddof=1) if values.shape[0] > 1 else np.zeros_like(mean)
    return mean, std

In [ ]:
from scipy.stats import wilcoxon


def save(fig, name):
    fig.savefig(OUT_BASE / 'images' / 'comparison' / f'{name}.svg', dpi=200, bbox_inches='tight')
    fig.savefig(OUT_BASE / 'images_png' / 'comparison' / f'{name}.png', dpi=200, bbox_inches='tight')
    fig.savefig(OUT_BASE / 'images_svg' / 'comparison' / f'{name}.svg', bbox_inches='tight')


def sig_marker(p):
    if np.isnan(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return ''


def paired_pvalues(old_arr, new_arr):
    pvals = np.full(len(METRICS), np.nan)
    for mi in range(len(METRICS)):
        o, n = old_arr[:, mi], new_arr[:, mi]
        mask = ~(np.isnan(o) | np.isnan(n))
        o_v, n_v = o[mask], n[mask]
        if len(o_v) < 2 or np.allclose(o_v, n_v):
            continue
        try:
            _, pvals[mi] = wilcoxon(o_v, n_v, zero_method='wilcox', alternative='two-sided')
        except ValueError:
            pass
    return pvals


def plot_old_vs_new_seeds(ax, old_arr, new_arr, title):
    pvals = paired_pvalues(old_arr, new_arr)
    labels = [
        f'{label} {sig_marker(p)}'.rstrip()
        for (label, _, _), p in zip(METRICS, pvals)
    ]
    x = np.arange(len(labels))
    width = 0.36
    old_mean, old_std = mean_std(old_arr)
    new_mean, new_std = mean_std(new_arr)
    ax.bar(x - width / 2, old_mean, width, yerr=old_std, color=OLD_COLOR,
           edgecolor='none', capsize=3, label=f'Old (grid search)',
           error_kw=dict(ecolor='black', lw=1.0))
    ax.bar(x + width / 2, new_mean, width, yerr=new_std, color=NEW_COLOR,
           edgecolor='none', capsize=3, label=f'New (backprop)',
           error_kw=dict(ecolor='black', lw=1.0))
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=10, rotation=30, ha='right')
    ax.set_ylim(0, 1.0)
    ax.set_yticks([0.0, 0.5, 1.0])
    # ax.set_title(title, fontsize=14)
    ax.yaxis.grid(True, linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)


fig_w = max(4.0, 0.6 * len(METRICS) + 0.5)
for ds in DATASETS:
    print(f'\n== {ds} ==')
    old_arr = load_seed_metrics(ds, OLD_KEY)
    new_arr = load_seed_metrics(ds, NEW_KEY)
    fig, ax = plt.subplots(figsize=(fig_w, 3.5))
    fig.patch.set_alpha(0.0)
    plot_old_vs_new_seeds(ax, old_arr, new_arr, DATASET_TITLES.get(ds, ds))
    ax.legend(loc='upper right', bbox_to_anchor=(1.0, 1.0), frameon=False, fontsize=10, ncols=2)
    plt.tight_layout()
    save(fig, f'{ds}_old_vs_new_hopf_seeds')
    plt.show()

In [ ]:
from scipy.stats import wilcoxon

# Paired Wilcoxon signed-rank test (old vs new) per metric, across seeds.
# Wilcoxon's minimum two-sided p-value depends on n: n=3 → 0.25, n=6 → 0.0625,
# n=7 → 0.0312, n=9 → 0.0039. With the 10 paired seeds wired up above we can
# reach significance at p<0.05; missing JSONs (e.g. a failed run) are NaN-masked.
print(f'Paired Wilcoxon signed-rank test: OLD ({OLD_KEY}) vs NEW ({NEW_KEY})')
print(f'Across {len(SEEDS)} seeds (per metric).\n')

for ds in DATASETS:
    print(f'== {DATASET_TITLES.get(ds, ds)} ==')
    old_arr = load_seed_metrics(ds, OLD_KEY)
    new_arr = load_seed_metrics(ds, NEW_KEY)
    header = f'{"metric":<24} {"old mean":>10} {"new mean":>10} {"new-old":>10} {"n":>4} {"W":>8} {"p":>8}'
    print(header)
    print('-' * len(header))
    for mi, (label, key, higher_better) in enumerate(METRICS):
        o = old_arr[:, mi]
        n = new_arr[:, mi]
        mask = ~(np.isnan(o) | np.isnan(n))
        o_v, n_v = o[mask], n[mask]
        if len(o_v) < 2 or np.allclose(o_v, n_v):
            stat, p = np.nan, np.nan
        else:
            try:
                stat, p = wilcoxon(o_v, n_v, zero_method='wilcox', alternative='two-sided')
            except ValueError:
                stat, p = np.nan, np.nan
        om, nm = np.nanmean(o), np.nanmean(n)
        print(f'{key:<24} {om:>10.4f} {nm:>10.4f} {nm - om:>+10.4f} {int(mask.sum()):>4d} {stat:>8.2f} {p:>8.3f}')
    print()

## How to regenerate the underlying JSON

Run the following sbatch jobs (3 seeds × 2 algorithms × 2 datasets = 12). Each writes one JSON to `results/runs/`.

```bash
for s in 42 43 44; do
  for ds_args in "--dataset-type lsd --lsd-data-dir data/lsd" \
                  "--dataset-type ts_young --data-path data/ts_young/ts_young_TR0.72.mat"; do
    sbatch -M gpu_cluster --gres=gpu:1 --time=02:00:00 --wrap=".venv/bin/python examples/train_models.py hopf-grid $ds_args --no-wandb --seed $s --run-suffix seed$s"
    sbatch -M gpu_cluster --gres=gpu:1 --time=02:00:00 --wrap=".venv/bin/python examples/train_models.py backprop --model hopf $ds_args --no-wandb --seed $s --run-suffix seed$s"
  done
done
```